# D322 — Olist Bronze-to-Silver Exercise with the Spark Catalog

Stage the Olist CSV data already available in HDFS as Bronze catalog tables, then create typed Silver Parquet tables. Run this notebook before D323–D326. This is an exercise: configuration and patterns are shown, but table schemas and transformation answers are not supplied.


## Architecture and scope

`HDFS raw CSV → olist_bronze external CSV tables → olist_silver managed Parquet tables`

Required datasets: customers, orders, order items, order payments, order reviews, products, and sellers. Deliberately omit geolocation and category translation. In Silver products, retain `product_category_name` but omit size/weight/description columns unless you justify their use.

Orders and order items should be partitioned by purchase `year` and `month`, not by day. A managed `saveAsTable` records metadata in the Spark catalog and stores files under the configured HDFS warehouse.


## 1. Start services

Before the notebook, start only HDFS and the Spark standalone master/worker. Do **not** start or configure an external Hive Metastore service. These notebooks use Spark's persistent session catalog with a local embedded Derby metadata store.

The default warehouse below is `hdfs:///user/hive/warehouse`. To use the course alternative, set `SPARK_SQL_WAREHOUSE=hdfs:///user/spark/warehouse` **before** creating the Spark session. Do not switch warehouses after tables have been created.

Catalog metadata persists locally under `~/.spark-catalog/d32_olist_metastore` by default. All D322–D326 notebooks must use the same `SPARK_CATALOG_DIR`. Embedded Derby permits only one active Spark driver to open this catalog, so stop the previous notebook's Spark session before starting another. If an older incompatible catalog exists, choose a new empty `SPARK_CATALOG_DIR`; do not point these notebooks at an old Hive metastore database.


In [ ]:
import os
import socket

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D32-Spark-SQL-Window-Functions")
    .master(master_url)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)


## 2. Locate existing raw CSVs in HDFS

Set `OLIST_RAW_HDFS` to the parent directory containing one dataset directory or file per CSV. Do not re-upload files that already exist. If your files are elsewhere, change only this variable.


In [ ]:
raw_root = os.environ.get("OLIST_RAW_HDFS", f"hdfs:///user/{os.environ['USER']}/olist/raw")
print("Raw root:", raw_root)

# TODO: List and inspect the seven required HDFS inputs with the Hadoop CLI.
# Confirm headers, file names, logical row counts, quoting, null representation,
# and whether review comments contain embedded newlines.


## 3. Create catalog databases

Create `olist_bronze` and `olist_silver` through `spark.sql`. Give each database an explicit HDFS `LOCATION` so storage ownership is visible. Suggested locations are `/user/spark/olist/bronze` and `/user/spark/olist/silver`; the Silver managed tables may instead inherit the configured warehouse.


In [ ]:
# TODO: CREATE DATABASE IF NOT EXISTS ... LOCATION ...
# TODO: SHOW DATABASES and DESCRIBE DATABASE EXTENDED for both databases.


## 4. Register Bronze external CSV tables

Create one external catalog table for each required CSV. Bronze must preserve the source values with minimal interpretation, preferably as strings, and point to the existing HDFS CSV location. Use Spark's CSV data source options for header, quoting, escaping, and multiline review text.

Required table names:

- `olist_bronze.olist_customers`
- `olist_bronze.olist_orders`
- `olist_bronze.olist_order_items`
- `olist_bronze.olist_order_payments`
- `olist_bronze.olist_order_reviews`
- `olist_bronze.olist_products`
- `olist_bronze.olist_sellers`

Do not register geolocation or category translation.


In [ ]:
# Pattern only — replace placeholders; do not copy it as a completed table.
# spark.sql(f"""
# CREATE TABLE olist_bronze.<table_name> (<all_required_raw_columns_as_STRING>)
# USING csv OPTIONS (path '<hdfs-path>', header 'true', quote '"', escape '"')
# """)

# TODO: Create all seven Bronze external tables.


## 5. Validate Bronze

For every table, record row count, column count, duplicate business keys, null/blank keys, and a small sample. Compare with the common Olist counts: customers 99,441; orders 99,441; items 112,650; payments 103,886; reviews 99,224; products 32,951; sellers 3,095. Investigate rather than hiding discrepancies.


In [ ]:
# TODO: Build and display a compact validation report for all seven Bronze tables.


## 6. Transform Bronze into typed Silver Parquet

Create typed, cleaned DataFrames using explicit casts (`timestamp`, integral, and decimal types), blank-to-null handling, and selected columns. Do not use schema inference as the final Silver contract.

Required partitioning:

- `olist_silver.olist_orders`: derive `purchase_year` and `purchase_month` from `order_purchase_timestamp`; partition by both.
- `olist_silver.olist_order_items`: obtain purchase year/month by joining to typed orders; partition by both.
- Other small dimensions/facts: do not partition unless you demonstrate a benefit.

Use Parquet and `saveAsTable`. Choose an idempotent rerun strategy and explain its effect before using `overwrite`.


In [ ]:
# API pattern only; fill in the transformation and table names.
# (
#     silver_df.write
#     .format("parquet")
#     .mode("overwrite")
#     .partitionBy("purchase_year", "purchase_month")
#     .saveAsTable("olist_silver.<table_name>")
# )

# TODO: Create all seven Silver managed Parquet tables.


## 7. Verify catalog metadata and HDFS files

Use Spark SQL to run `SHOW TABLES`, `DESCRIBE EXTENDED`, `SHOW PARTITIONS`, and row-count checks. Confirm Bronze provider/location and Silver provider/location. Then use `hdfs dfs -ls -R` to confirm Parquet files and year/month directories physically exist.


In [ ]:
# TODO: Catalog checks.
# TODO: HDFS listing via a notebook bash cell or terminal.


## 8. Data-quality gates

Add executable assertions for expected counts, primary business-key uniqueness where applicable, orphan joins, valid state length, nonnegative money, timestamp cast failures, and valid order partition columns. Explain any review-row discrepancy caused by multiline CSV handling.


In [ ]:
# TODO: Add assertions. A failed gate must stop the notebook.


## Submission

Submit the completed notebook plus a short table inventory containing catalog name, provider, grain, partition columns, row count, and HDFS location. Do not include geolocation or category-translation data.
